In [1]:
"""
PSB Shield — Model Trainer
Trains two Random Forest classifiers specifically on SBI YONO fraud patterns.

Run once:
    python train_model.py

Outputs:
    models/url_model.pkl
    models/apk_model.pkl
    models/label_encoder_url.pkl
    models/label_encoder_apk.pkl
"""

import os, random, joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

os.makedirs("models", exist_ok=True)
random.seed(42)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# SBI / YONO SPECIFIC CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

OFFICIAL_DOMAINS  = ["sbi.co.in", "onlinesbi.sbi", "yonobusiness.sbi", "retail.onlinesbi.sbi"]
REAL_PACKAGE      = "com.sbi.lotusintouch"

# Known fake domain patterns targeting YONO users
FAKE_DOMAINS = [
    "yono-sbi.xyz", "sbi-yono.top", "sbiyono.tk",
    "onlinesbi.sbi.bank.inn", "yonosbi.online",
    "sbi-kyc-update.com", "sbikyc.xyz",
    "yono.apk-download.xyz", "sbi-reward.top",
    "sbimobile-update.tk", "yono-sbi-login.cf",
    "onlinesbi.sbi.net.in.xyz", "sbi.co.in.kyc.xyz",
    "sbionline-verify.club", "yono.sbi.account.inn",
    "sbi-alert.tk", "sbiyono-update.live",
    "yono.fake-sbi.xyz", "sbi-otp.top",
    "sbimobile.pw", "sbibank-kyc.online",
]

FAKE_PACKAGES = [
    "com.sbi.yono.update", "com.yono.sbi.fake",
    "com.sbionline.login", "com.sbi.yono.new",
    "in.sbi.yono", "com.fakebank.sbi",
    "com.sbi.yono.v2", "org.sbi.yono",
    "com.sbi.mobile.fake", "com.yono.banking",
    "net.sbi.lotusintouch", "com.sbimobile.update",
    "com.sbi.kyc.update", "in.yonosbi.app",
    "com.android.sbi.yono", "com.sbi.reward",
]

SUSPICIOUS_PACKAGES = [
    "com.sbi.lotusintouch.beta", "com.sbiyono.official",
    "com.sbi.yono.lite", "com.statebankofindiaapp",
    "com.sbigroup.yono", "in.sbi.banking",
]

URGENCY_PHRASES = [
    "kyc expire", "account block", "urgent verify",
    "click immediately", "otp required", "reward claim",
    "account suspend", "update now", "deactivate",
]

SUSPICIOUS_TLDS = [
    ".xyz", ".tk", ".top", ".gq", ".ml", ".cf",
    ".pw", ".club", ".inn", ".online", ".site",
    ".live", ".download", ".win", ".party",
]

SAFE_PERMS = [
    "INTERNET", "ACCESS_NETWORK_STATE", "CAMERA",
    "USE_BIOMETRIC", "USE_FINGERPRINT", "VIBRATE",
    "RECEIVE_BOOT_COMPLETED", "ACCESS_WIFI_STATE",
]

DANGEROUS_PERMS = [
    "READ_SMS", "RECEIVE_SMS", "SEND_SMS",
    "READ_CONTACTS", "READ_CALL_LOG",
    "SYSTEM_ALERT_WINDOW", "BIND_ACCESSIBILITY_SERVICE",
    "RECORD_AUDIO", "READ_PHONE_STATE",
    "PROCESS_OUTGOING_CALLS",
]


# ─────────────────────────────────────────────────────────────────────────────
# URL FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_url_features(url: str) -> list:
    """
    Extract 15 features from a URL, all tuned to SBI YONO fraud patterns.
    Returns a list of numeric features.
    """
    lower = url.lower()

    # 1. Has official SBI domain
    f1 = int(any(d in lower for d in OFFICIAL_DOMAINS))

    # 2. SBI/YONO brand used in unofficial domain
    has_brand = any(k in lower for k in ["sbi", "yono", "sbionline", "onlinesbi"])
    f2 = int(has_brand and not f1)

    # 3. Suspicious TLD
    f3 = int(any(t in lower for t in SUSPICIOUS_TLDS))

    # 4. URL shortener
    f4 = int(any(s in lower for s in ["bit.ly","tinyurl","t.co","goo.gl","ow.ly","rb.gy","cutt.ly"]))

    # 5. Raw IP address
    import re
    f5 = int(bool(re.search(r'https?://\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', lower)))

    # 6. Direct APK download
    f6 = int(".apk" in lower)

    # 7. Urgency / social engineering words
    urgency = ["urgent","expire","kyc","block","suspend","verify","immediate","otp","reward","freeze"]
    f7 = sum(1 for w in urgency if w in lower)

    # 8. Subdomain depth
    try:
        dp = re.findall(r'https?://([^/]+)', lower)
        f8 = dp[0].count('.') if dp else 0
    except:
        f8 = 0

    # 9. Is HTTPS
    f9 = int(lower.startswith("https://"))

    # 10. Domain length
    try:
        dp = re.findall(r'https?://([^/]+)', lower)
        f10 = len(dp[0]) if dp else 0
    except:
        f10 = 0

    # 11. Digits in domain
    try:
        dp = re.findall(r'https?://([^/]+)', lower)
        f11 = sum(c.isdigit() for c in (dp[0] if dp else ""))
    except:
        f11 = 0

    # 12. Hyphens in domain
    try:
        dp = re.findall(r'https?://([^/]+)', lower)
        f12 = (dp[0] if dp else "").count('-')
    except:
        f12 = 0

    # 13. Total URL length
    f13 = len(url)

    # 14. YONO keyword specifically
    f14 = int("yono" in lower)

    # 15. Has "sbi" in domain but not official
    f15 = int(f2 and "sbi" in lower)

    return [f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15]

URL_FEATURE_NAMES = [
    "has_official_sbi_domain", "sbi_brand_in_unofficial",
    "suspicious_tld", "url_shortener", "raw_ip_address",
    "apk_download_link", "urgency_word_count", "subdomain_depth",
    "is_https", "domain_length", "digits_in_domain",
    "hyphens_in_domain", "url_total_length", "has_yono_keyword",
    "sbi_in_unofficial_domain",
]


# ─────────────────────────────────────────────────────────────────────────────
# APK FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def extract_apk_features(package: str, permissions: list, cert: str, embedded_urls: list) -> list:
    """
    Extract 12 features from APK metadata.
    All features are SBI YONO specific.
    """
    perms_upper = [p.upper() for p in permissions]

    # 1. Exact match with real YONO package
    f1 = int(package == REAL_PACKAGE)

    # 2. SBI/YONO in package name but not real package
    f2 = int(("sbi" in package.lower() or "yono" in package.lower()) and package != REAL_PACKAGE)

    # 3. SMS permissions
    f3 = int(any(p in perms_upper for p in ["READ_SMS","RECEIVE_SMS","SEND_SMS"]))

    # 4. Overlay permission (screen overlay — used for UI spoofing)
    f4 = int("SYSTEM_ALERT_WINDOW" in perms_upper)

    # 5. Accessibility service (keylogger risk)
    f5 = int("BIND_ACCESSIBILITY_SERVICE" in perms_upper)

    # 6. Call log access
    f6 = int("READ_CALL_LOG" in perms_upper or "PROCESS_OUTGOING_CALLS" in perms_upper)

    # 7. Audio recording
    f7 = int("RECORD_AUDIO" in perms_upper)

    # 8. Self-signed or missing certificate
    f8 = int("self" in cert.lower() or "missing" in cert.lower() or "unverified" in cert.lower())

    # 9. Suspicious embedded URLs (non-SBI)
    bad_urls = [u for u in embedded_urls if not any(d in u for d in OFFICIAL_DOMAINS)]
    f9 = min(len(bad_urls), 5)

    # 10. Total dangerous permission count
    f10 = sum(1 for p in perms_upper if p in [d.upper() for d in DANGEROUS_PERMS])

    # 11. Contacts access
    f11 = int("READ_CONTACTS" in perms_upper)

    # 12. Phone state access
    f12 = int("READ_PHONE_STATE" in perms_upper)

    return [f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12]

APK_FEATURE_NAMES = [
    "is_correct_package", "sbi_yono_in_fake_package",
    "sms_permission", "overlay_permission",
    "accessibility_permission", "call_log_permission",
    "audio_record_permission", "self_signed_cert",
    "suspicious_url_count", "total_dangerous_perms",
    "contacts_permission", "phone_state_permission",
]


# ─────────────────────────────────────────────────────────────────────────────
# SYNTHETIC DATA GENERATION — SBI YONO SPECIFIC
# ─────────────────────────────────────────────────────────────────────────────

def gen_url_data(n_per_class=400):
    X, y = [], []

    # ── Class 0: SAFE — official SBI domains ─────────────────────────────────
    safe_urls = [
        "https://onlinesbi.sbi/login",
        "https://retail.onlinesbi.sbi/",
        "https://yonobusiness.sbi/corp/AuthenticationController",
        "https://sbi.co.in/web/personal-banking",
        "https://onlinesbi.sbi/sbijava/login.jsp",
        "https://sbi.co.in/portal/web/personal-banking/home",
    ]
    for _ in range(n_per_class):
        url = random.choice(safe_urls)
        # Add minor variation
        if random.random() > 0.5:
            url += "?ref=" + str(random.randint(1000,9999))
        X.append(extract_url_features(url))
        y.append("SAFE")

    # ── Class 1: SUSPICIOUS — borderline cases ────────────────────────────────
    suspicious_urls = [
        "https://sbi-customercare.co.in/help",
        "https://sbionline.net.in/portal",
        "https://yono.sbi.bank/login",
        "https://onlinesbi.net/customer",
        "http://sbi.co.in.verify-kyc.com/",
        "https://sbimobile.org/login",
        "https://sbi-alert.info/update",
    ]
    for _ in range(n_per_class):
        url = random.choice(suspicious_urls)
        X.append(extract_url_features(url))
        y.append("SUSPICIOUS")

    # ── Class 2: PHISHING — known SBI YONO fraud patterns ────────────────────
    phishing_urls = [
        # Fake domain + APK download
        "http://yono-sbi.apk-download.xyz/YONO_SBI_update.apk",
        "http://sbi-kyc.tk/download/yono.apk",
        "https://sbiyono.online/install.apk",
        # IP-based
        "http://192.168.1.104/sbi/login",
        "http://103.21.58.96/yono/otp",
        "http://45.33.32.156/sbi-kyc",
        # URL shorteners hiding SBI phishing
        "https://bit.ly/sbi-kyc-urgent",
        "https://tinyurl.com/yono-update",
        "https://rb.gy/sbi-reward",
        # Fake domains with urgency
        "http://onlinesbi.sbi.bank.inn/kyc-expire-verify",
        "https://sbi-otp.top/urgent-kyc-update",
        "http://yonosbi-login.cf/account-block",
        "https://sbimobile-update.tk/reward-claim",
        "http://sbi-alert.xyz/freeze-account",
        "https://yono.fake-sbi.top/otp-verify",
    ]
    for _ in range(n_per_class):
        url = random.choice(phishing_urls)
        X.append(extract_url_features(url))
        y.append("PHISHING")

    return np.array(X), np.array(y)


def gen_apk_data(n_per_class=400):
    X, y = [], []

    # ── Class 0: LEGITIMATE — real YONO SBI app ───────────────────────────────
    for _ in range(n_per_class):
        perms = random.sample(SAFE_PERMS, k=random.randint(3,6))
        feats = extract_apk_features(
            package       = REAL_PACKAGE,
            permissions   = perms,
            cert          = "Verified (State Bank of India, CN=SBI)",
            embedded_urls = ["https://retail.onlinesbi.sbi", "https://sbi.co.in"]
        )
        X.append(feats)
        y.append("LEGITIMATE")

    # ── Class 1: SUSPICIOUS ────────────────────────────────────────────────────
    for _ in range(n_per_class):
        pkg   = random.choice(SUSPICIOUS_PACKAGES)
        perms = random.sample(SAFE_PERMS, k=random.randint(2,5)) + \
                random.sample(DANGEROUS_PERMS, k=random.randint(1,2))
        feats = extract_apk_features(
            package       = pkg,
            permissions   = perms,
            cert          = random.choice(["Self-signed", "Verified (unknown CA)"]),
            embedded_urls = ["https://sbi.co.in"] + \
                            (["http://unknown-server.com"] if random.random()>0.5 else [])
        )
        X.append(feats)
        y.append("SUSPICIOUS")

    # ── Class 2: FAKE_MALICIOUS — known fake YONO patterns ────────────────────
    for _ in range(n_per_class):
        pkg   = random.choice(FAKE_PACKAGES)
        perms = random.sample(SAFE_PERMS, k=random.randint(1,3)) + \
                random.sample(DANGEROUS_PERMS, k=random.randint(3,7))
        bad_urls = [
            "http://192.168.1.104/steal_otp",
            "http://malware-c2.xyz/upload",
            "http://sbi-fake-server.tk/credentials",
            "http://45.33.32.156/keylog",
        ]
        feats = extract_apk_features(
            package       = pkg,
            permissions   = perms,
            cert          = "Self-signed or missing",
            embedded_urls = random.sample(bad_urls, k=random.randint(1,3))
        )
        X.append(feats)
        y.append("FAKE_MALICIOUS")

    return np.array(X), np.array(y)


# ─────────────────────────────────────────────────────────────────────────────
# TRAIN
# ─────────────────────────────────────────────────────────────────────────────

def train_url_model():
    print("\n" + "═"*55)
    print("  TRAINING URL CLASSIFIER")
    print("  Target: SBI YONO phishing links")
    print("═"*55)

    X, y = gen_url_data(n_per_class=500)
    print(f"  Dataset: {len(X)} samples — {dict(zip(*np.unique(y, return_counts=True)))}")

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

    model = RandomForestClassifier(
        n_estimators  = 200,
        max_depth     = 10,
        min_samples_split = 4,
        class_weight  = "balanced",
        random_state  = 42,
        n_jobs        = -1,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Feature importance
    print("  Top features by importance:")
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    for i in range(min(8, len(URL_FEATURE_NAMES))):
        idx = indices[i]
        print(f"    {i+1}. {URL_FEATURE_NAMES[idx]:<35} {importances[idx]:.3f}")

    joblib.dump(model, "models/url_model.pkl")
    joblib.dump(le,    "models/label_encoder_url.pkl")
    print("\n  ✓ Saved → models/url_model.pkl")
    return model, le


def train_apk_model():
    print("\n" + "═"*55)
    print("  TRAINING APK CLASSIFIER")
    print("  Target: Fake YONO SBI app detection")
    print("═"*55)

    X, y = gen_apk_data(n_per_class=500)
    print(f"  Dataset: {len(X)} samples — {dict(zip(*np.unique(y, return_counts=True)))}")

    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

    model = RandomForestClassifier(
        n_estimators  = 200,
        max_depth     = 12,
        min_samples_split = 3,
        class_weight  = "balanced",
        random_state  = 42,
        n_jobs        = -1,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    print("  Top features by importance:")
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    for i in range(min(8, len(APK_FEATURE_NAMES))):
        idx = indices[i]
        print(f"    {i+1}. {APK_FEATURE_NAMES[idx]:<35} {importances[idx]:.3f}")

    joblib.dump(model, "models/apk_model.pkl")
    joblib.dump(le,    "models/label_encoder_apk.pkl")
    print("\n  ✓ Saved → models/apk_model.pkl")
    return model, le


# ─────────────────────────────────────────────────────────────────────────────
# QUICK TEST — sanity check after training
# ─────────────────────────────────────────────────────────────────────────────

def quick_test(url_model, url_le, apk_model, apk_le):
    print("\n" + "═"*55)
    print("  QUICK SANITY TEST")
    print("═"*55)

    test_urls = [
        ("https://onlinesbi.sbi/login",               "should be SAFE"),
        ("http://yono-sbi.apk-download.xyz/yono.apk", "should be PHISHING"),
        ("https://bit.ly/sbi-kyc-urgent",             "should be PHISHING"),
        ("http://192.168.1.1/sbi-login",              "should be PHISHING"),
        ("https://sbi.co.in/portal",                  "should be SAFE"),
        ("https://sbionline.net.in/portal",           "should be SUSPICIOUS"),
    ]

    print("\n  URL Tests:")
    for url, expected in test_urls:
        feats = np.array([extract_url_features(url)])
        pred  = url_le.inverse_transform(url_model.predict(feats))[0]
        proba = url_model.predict_proba(feats)[0]
        conf  = int(max(proba)*100)
        ok    = "✓" if expected.split()[-1].upper() in pred else "✗"
        print(f"    {ok} [{pred:<12}] {conf}% — {url[:55]}")

    test_apks = [
        (REAL_PACKAGE, SAFE_PERMS[:4], "Verified (SBI)",         [],                           "should be LEGITIMATE"),
        ("com.sbi.yono.update", DANGEROUS_PERMS[:5]+SAFE_PERMS[:2], "Self-signed", ["http://malware.xyz"], "should be FAKE_MALICIOUS"),
        ("com.sbiyono.official", SAFE_PERMS[:3]+DANGEROUS_PERMS[:2], "Self-signed", [],          "should be SUSPICIOUS"),
    ]

    print("\n  APK Tests:")
    for pkg, perms, cert, urls, expected in test_apks:
        feats = np.array([extract_apk_features(pkg, perms, cert, urls)])
        pred  = apk_le.inverse_transform(apk_model.predict(feats))[0]
        proba = apk_model.predict_proba(feats)[0]
        conf  = int(max(proba)*100)
        ok    = "✓" if expected.split()[-1].upper() in pred else "✗"
        print(f"    {ok} [{pred:<16}] {conf}% — {pkg}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("\n  PSB Shield — Model Training")
    print("  Specifically targeting SBI YONO fraud patterns\n")

    url_model, url_le = train_url_model()
    apk_model, apk_le = train_apk_model()
    quick_test(url_model, url_le, apk_model, apk_le)

    print("\n" + "═"*55)
    print("  DONE. Models saved to /models folder.")
    print("  Now restart main.py to use them.")
    print("═"*55 + "\n")


  PSB Shield — Model Training
  Specifically targeting SBI YONO fraud patterns


═══════════════════════════════════════════════════════
  TRAINING URL CLASSIFIER
  Target: SBI YONO phishing links
═══════════════════════════════════════════════════════
  Dataset: 1500 samples — {np.str_('PHISHING'): np.int64(500), np.str_('SAFE'): np.int64(500), np.str_('SUSPICIOUS'): np.int64(500)}

  Classification Report:
              precision    recall  f1-score   support

    PHISHING       1.00      1.00      1.00       100
        SAFE       1.00      1.00      1.00       100
  SUSPICIOUS       1.00      1.00      1.00       100

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

  Top features by importance:
    1. has_official_sbi_domain             0.132
    2. urgency_word_count                  0.130
    3. sbi_brand_in_unofficial             0.117
    4. sbi_in_unofficial_doma